# 🏥 Benchmark Evaluations Notebook 📈
  
**Goal**: Load ranking results for different retrieval methods, generate query/response pairs, compute per-query metrics (NDCG@10, NDCG@20, Recall@10), aggregate averages, select the best method, and upload to Azure AI Foundry.  
Phases:  
1. 📦 **Preprocess** (load data & build pairs)  
2. ⚙️ **Evaluate** (compute metrics)  
3. ➡️ **Post-process** (aggregate & upload)  

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import json

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import evaluate, Evaluator
from beir.retrieval.evaluation import EvaluateRetrieval

# ─── 0. Setup Azure AI Foundry Connection ──────────────────────────────────────
load_dotenv()
AZ_CONN = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")
_, AZ_SUBSCRIPTION_ID, AZ_RESOURCE_GROUP, AI_FOUNDRY_PROJECT_NAME = AZ_CONN.split(";")

credential = DefaultAzureCredential()
ai_project_client = AIProjectClient.from_connection_string(
    credential=credential,
    conn_str=AZ_CONN
)
print(f"✅ Connected to Azure AI Project: {AI_FOUNDRY_PROJECT_NAME}")

# ─── 1. Preprocess: build eval_data.jsonl ───────────────────────────────────────
EVAL_DIR      = Path("evals/benchmark/medindexer")
qrels_path    = EVAL_DIR / "qrels.jsonl"
ranking_files = {
    "hybrid_semantic": EVAL_DIR / "rankings-hybrid-semantic.jsonl",
    "hybrid":          EVAL_DIR / "rankings-hybrid.jsonl",
    "keyword":         EVAL_DIR / "rankings-keyword.jsonl",
    "vector":          EVAL_DIR / "rankings-vector.jsonl",
}

# load qrels into dict[query][doc]=rel
def load_qrels(path: Path):
    q = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            q.setdefault(r["query"], {})[r["document"]] = int(r["relevant"])
    return q

qrels = load_qrels(qrels_path)

# load rankings into dict[method][query]={doc:score}
rankings = {}
for name, fp in ranking_files.items():
    run = {}
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            ent = json.loads(line)
            run[ent["query"]] = ent["ranking"]
    rankings[name] = run

# write eval_data.jsonl: each record has query, response (JSON list), ground_truth (JSON dict)
eval_data_path = EVAL_DIR / "eval_data.jsonl"
with open(eval_data_path, "w", encoding="utf-8") as fout:
    for method, run in rankings.items():
        for qid, ranking in run.items():
            # sort docs by descending score
            docs_sorted = [doc for doc, _ in sorted(ranking.items(), key=lambda x: x[1], reverse=True)]
            rec = {
                "query":         qid,
                "response":      json.dumps(docs_sorted),
                "ground_truth":  json.dumps(qrels.get(qid, {}))
            }
            fout.write(json.dumps(rec) + "\n")
print(f"📦 Preprocessed eval data written to {eval_data_path}")

# ─── 2. Custom Evaluators ───────────────────────────────────────────────────────
class NDCGEvaluator(Evaluator):
    def __init__(self, k: int):
        self.k = k
    def __call__(self, *, response: str, ground_truth: str, **kwargs):
        # parse
        resp = json.loads(response)
        gt   = json.loads(ground_truth)
        # build run and qrels for BEIR
        run   = {resp[i]: i for i in range(len(resp))}
        qrels = {qid: gt}
        # evaluate
        results = EvaluateRetrieval().evaluate(qrels, {qid: run}, k_values=[self.k])
        score   = results[0].get(f"NDCG@{self.k}", 0.0)
        return {f"NDCG@{self.k}": score}

class RecallEvaluator(Evaluator):
    def __init__(self, k: int):
        self.k = k
    def __call__(self, *, response: str, ground_truth: str, **kwargs):
        resp = json.loads(response)[:self.k]
        gt   = set(json.loads(ground_truth).keys())
        hits = len([doc for doc in resp if doc in gt])
        total= len(gt) or 1
        return {f"Recall@{self.k}": hits/total}

# ─── 3. Evaluate via AI Foundry ────────────────────────────────────────────────
evaluate(
    evaluation_name="search-benchmark",
    data=str(eval_data_path),
    evaluators={
        "ndcg_3":    NDCGEvaluator(3),
        "ndcg_10":   NDCGEvaluator(10),
        "recall_10": RecallEvaluator(10),
    },
    azure_ai_project=ai_project_client,
    evaluator_config={
        "ndcg_3":    {"column_mapping": {"response": "${data.response}", "ground_truth": "${data.ground_truth}"}},
        "ndcg_10":   {"column_mapping": {"response": "${data.response}", "ground_truth": "${data.ground_truth}"}},
        "recall_10": {"column_mapping": {"response": "${data.response}", "ground_truth": "${data.ground_truth}"}},
    }
)
print("✅ Evaluation complete and metrics registered in AI Foundry")

